# Validation per user activity group (W10)

Popularity, MF, TF-IDF CBF and Embedding CBF are evaluated on validation with the shared protocol
(`src/protocol.py` through `src/shared_eval.py`) and `src/evaluate.py`. Each model produces its own
top-10 lists through its existing `recommend` method; no model code or parameter is changed.
Only `interactions_train.csv` and `interactions_validation.csv` are read; the test split is not used.

Sections:
1. Protocol and evaluable users
2. Fit the four models
3. Overall metrics
4. Metrics per user activity group (RQ2)
5. Leakage checks from `src/leakage_check.py`

In [1]:
import json
import platform
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.shared_eval import build_protocol, evaluable_users_rule, summarize, check_top_k
from src.protocol import protocol_counts, compare_with_official
from src.evaluate import sanity_check_metrics
from src import user_groups
from src.leakage_check import (LeakageError, run_static_checks, check_recommendations,
                               check_same_candidates, check_features)

SEED, K = 42, 10
random.seed(SEED)
np.random.seed(SEED)
DATA_DIR = str(ROOT / 'data') + '/'
RULE = evaluable_users_rule()

import scipy, sklearn, torch, sentence_transformers
display(pd.Series({
    'evaluable_users setting (configs/evaluation.json)': RULE,
    'python': platform.python_version(),
    'numpy': np.__version__, 'pandas': pd.__version__, 'scipy': scipy.__version__,
    'scikit-learn': sklearn.__version__, 'torch': torch.__version__,
    'sentence-transformers': sentence_transformers.__version__,
}, name='value').to_frame())

,value
evaluable_users setting (configs/evaluation.json),official
python,3.11.4
numpy,1.24.3
pandas,1.5.3
scipy,1.10.1
scikit-learn,1.3.0
torch,2.15.0+cpu
sentence-transformers,6.2.0


## 1. Protocol and evaluable users

The list of evaluable users is chosen with one setting, `evaluable_users` in `configs/evaluation.json`:
`official` uses `src.protocol.build_protocol` (at least one train rating >= 4 and at least one relevant
validation item); `group_rule` keeps the earlier rule without the train filter. All four models use the
same `relevant`, `seen` and candidate set.

In [2]:
data = build_protocol(DATA_DIR)
train, val, cand_set, seen, relevant = data
counts = protocol_counts(train, val, cand_set, seen, relevant)
official = compare_with_official(relevant, DATA_DIR)
display(pd.Series({**counts, **{f'official file: {k}': v for k, v in official.items()}},
                  name='value').to_frame())
if RULE == 'official':
    assert official['identical'], 'evaluable users differ from the official file'
sanity_check_metrics(verbose=False)

,value
train_interactions,427957
validation_interactions,41284
train_users,54483
train_items,22753
candidate_items_warm,22753
candidates_not_seen_in_train,0
evaluable_users,11659
relevant_pairs,24202
min_rating_relevant,4
top_k,10


True

## 2. Fit the four models

Parameters are the ones used in `notebooks/validation_eval.ipynb`, `outputs/configs/tfidf_cbf.json` and
`outputs/configs/semantic_cbf.json`. The embedding model reads the cached MiniLM vectors in
`outputs/embeddings/` (ignored by git). MF takes a few minutes to fit.

In [3]:
models, cache_info = user_groups.build_models(train, cand_set)
display(pd.DataFrame({
    'model': list(models),
    'parameters': [json.dumps({'min_rating': 4}), json.dumps(user_groups.MF_PARAMS),
                   json.dumps(user_groups.TFIDF_PARAMS),
                   json.dumps({**user_groups.EMB_PARAMS, 'encoder': 'all-MiniLM-L6-v2'})],
}).set_index('model'))

,parameters
model,
Popularity,"{""min_rating"": 4}"
MF,"{""n_factors"": 32, ""n_epochs"": 10, ""learning_ra..."
TF-IDF,"{""stop_words"": ""english"", ""min_df"": 2, ""max_df..."
Embedding,"{""min_rating"": 4, ""encoder"": ""all-MiniLM-L6-v2""}"


## 3. Overall metrics

Per-user metrics come from `src.evaluate.evaluate_per_user`; the overall row is summed and divided by n
as in `src.evaluate.evaluate_model`. The same call writes `outputs/metrics/user_groups_val.json`.
The last columns compare with `outputs/metrics/validation_results.csv` (Diana's shared run).

In [4]:
result, table, per_user, groups, recs = user_groups.run(models=models, data=data, save=True)

shared = pd.read_csv(ROOT / 'outputs' / 'metrics' / 'validation_results.csv').set_index('Model')
overall = table[table['group'] == 'all'].set_index('model').drop(columns='group')
overall['unique items in top-10'] = pd.Series(result['unique_items_in_top10'])
for m in ['Recall@10', 'NDCG@10', 'HitRate@10']:
    overall[f'{m} diff vs validation_results.csv'] = overall[m] - shared[m].reindex(overall.index)
overall = overall[['n_users', 'Recall@10', 'NDCG@10', 'HitRate@10', 'unique items in top-10',
                   'Recall@10 diff vs validation_results.csv', 'NDCG@10 diff vs validation_results.csv',
                   'HitRate@10 diff vs validation_results.csv']]
display(overall.style.format({'Recall@10': '{:.6f}', 'NDCG@10': '{:.6f}', 'HitRate@10': '{:.6f}',
                              'Recall@10 diff vs validation_results.csv': '{:.1e}',
                              'NDCG@10 diff vs validation_results.csv': '{:.1e}',
                              'HitRate@10 diff vs validation_results.csv': '{:.1e}'}))

,n_users,Recall@10,NDCG@10,HitRate@10,unique items in top-10,Recall@10 diff vs validation_results.csv,NDCG@10 diff vs validation_results.csv,HitRate@10 diff vs validation_results.csv
model,,,,,,,,
Popularity,11659,0.013499,0.009222,0.031049,18,9.0e-17,4.7e-17,5.9e-17
MF,11659,0.004289,0.002762,0.009263,3620,4.3e-17,2.5e-17,2.8e-17
TF-IDF,11659,0.012612,0.007353,0.021100,16093,3.5e-17,1.1e-05,1.0e-17
Embedding,11659,0.008686,0.005228,0.014924,14154,8.0e-17,7.6e-17,2.4e-17


The TF-IDF NDCG@10 differs from `validation_results.csv` by about 1e-5. One evaluable user has two
candidate items with exactly equal TF-IDF scores at ranks 1 and 2, one of them relevant; the order of
that tie depends on the library versions used, which moves the hit between rank 1 and rank 2.

In [5]:
tfidf = models['TF-IDF']
tied = []
for u, rel in relevant.items():
    profile = tfidf.user_profile(u)
    if profile is None:
        continue
    r = recs['TF-IDF'][u]
    hits = [i for i, x in enumerate(r) if x in rel]
    if not hits:
        continue
    scores = tfidf.cand_matrix @ profile
    for x in seen.get(u, ()):
        if x in tfidf.cand_pos:
            scores[tfidf.cand_pos[x]] = -np.inf
    for i in hits:
        s = scores[tfidf.cand_pos[r[i]]]
        if s > 0 and (scores == s).sum() > 1:
            tied.append({'user_id': u, 'hit rank': i + 1, 'score': s,
                         'ranks with this score': [j + 1 for j, y in enumerate(r)
                                                   if scores[tfidf.cand_pos[y]] == s],
                         'n relevant': len(rel)})
display(pd.DataFrame(tied))

,user_id,hit rank,score,ranks with this score,n relevant
0,AEN56HWRCVG6ZTCUJM54MXPYQSDQ,1,0.183583,"[1, 2]",5


## 4. Metrics per user activity group (RQ2)

Users are labelled by their number of train interactions (all ratings) with the bounds in
`configs/user_groups.json`: rendah <= 5, sedang 6-8, tinggi >= 9. The groups only break down the
metrics above; they do not filter the evaluation.

In [6]:
bounds = pd.DataFrame(result['group_bounds']).set_index('label')
bounds = bounds.astype('Int64').astype(str).replace('<NA>', '-')
bounds.columns = ['min train interactions', 'max train interactions']
group_counts = pd.DataFrame(result['evaluable_users_per_group']).T
group_counts.insert(0, 'train users', pd.Series(result['n_train_users_per_group']))
group_counts['share of evaluable users'] = (group_counts['n_evaluable_users']
                                            / result['n_evaluable_users']).map('{:.1%}'.format)
display(bounds.join(group_counts))

,min train interactions,max train interactions,train users,n_evaluable_users,n_without_positive_train_item,share of evaluable users
label,,,,,,
rendah,-,5,22117,6105,0,52.4%
sedang,6,8,18197,2364,0,20.3%
tinggi,9,-,14169,3190,0,27.4%


In [7]:
order_groups = ['rendah', 'sedang', 'tinggi', 'all']
order_models = ['Popularity', 'MF', 'TF-IDF', 'Embedding']
long = table.set_index(['group', 'model']).loc[[(g, m) for g in order_groups for m in order_models]]
display(long.style.format({'Recall@10': '{:.6f}', 'NDCG@10': '{:.6f}', 'HitRate@10': '{:.6f}'}))

In [8]:
wide = table.pivot(index='group', columns='model', values=['NDCG@10', 'Recall@10', 'HitRate@10'])
wide = wide.reindex(index=order_groups,
                    columns=pd.MultiIndex.from_product([['NDCG@10', 'Recall@10', 'HitRate@10'],
                                                        order_models]))
wide.insert(0, ('n_users', ''), table.drop_duplicates('group').set_index('group')['n_users'])
wide[('NDCG@10', 'Embedding - TF-IDF')] = wide[('NDCG@10', 'Embedding')] - wide[('NDCG@10', 'TF-IDF')]
display(wide.style.format('{:.6f}', subset=[c for c in wide.columns if c[0] != 'n_users']))

## 5. Leakage checks

Checks from `src/leakage_check.py` (Diana): static checks on the split, the features used by the models,
the candidate universe of each model, and every recommendation list (only warm candidates, no item the
user saw in train). `check_top_k` adds the length and uniqueness check (exactly 10 unique items).

In [9]:
static = run_static_checks(train, val, relevant, cand_set, root=str(ROOT))

# Columns read by the models: interaction columns and the item text column.
features_ok = check_features(list(train.columns) + ['item_id', 'content_text_basic'])

universes = {
    'Popularity (fit input)': set(cand_set),
    'MF': set(models['MF'].candidate_items),
    'TF-IDF': set(models['TF-IDF'].candidate_items),
    'Embedding': set(models['Embedding'].candidate_items),
}
same_candidates = check_same_candidates(universes)

rows = []
for name, model_recs in recs.items():
    try:
        rec_ok = check_recommendations(model_recs, cand_set, seen)
        message = ''
    except LeakageError as e:
        rec_ok, message = False, str(e)
    rows.append({'model': name, 'users checked': len(model_recs),
                 'check_recommendations': rec_ok, **check_top_k(model_recs, cand_set, seen, K),
                 'message': message})
leak = pd.DataFrame(rows).set_index('model')

display(pd.Series({**{f'static: {k}': v for k, v in static.items()},
                   'check_features': features_ok,
                   'check_same_candidates (MF, TF-IDF, Embedding, Popularity input)': same_candidates,
                   'Popularity items that can be recommended': len(models['Popularity'].ranked_items)},
                  name='result').to_frame())
display(leak)
total = int(leak[['wrong_length', 'duplicate_items', 'seen_items', 'non_candidate_items']].to_numpy().sum())
assert total == 0 and leak['check_recommendations'].all()
print('Total violations:', total)

,result
static: temporal_split,True
static: no_pair_overlap,True
static: relevant_in_candidates,True
static: test_unused,True
check_features,True
"check_same_candidates (MF, TF-IDF, Embedding, Popularity input)",True
Popularity items that can be recommended,22672


,users checked,check_recommendations,wrong_length,duplicate_items,seen_items,non_candidate_items,message
model,,,,,,,
Popularity,11659,True,0,0,0,0,
MF,11659,True,0,0,0,0,
TF-IDF,11659,True,0,0,0,0,
Embedding,11659,True,0,0,0,0,


Total violations: 0
